# Sales Data Analysis and Reporting

## Project Overview
This project analyzes retail transaction data from a multi-year retail chain dataset to uncover customer purchasing patterns, revenue trends, and marketing campaign effectiveness. The pipeline covers data cleaning, preparation, exploratory analysis, RFM-based customer segmentation, and visual reporting.

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats

# Consistent plot style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 110

In [ ]:
transactions = pd.read_csv("../data/Retail_Data_Transactions.csv")
response     = pd.read_csv("../data/Retail_Data_Response.csv")

print("Transactions:", transactions.shape)
print("Response:    ", response.shape)
transactions.head()

In [ ]:
response.head()

## 2. Data Cleaning

We check for missing values, duplicates, and fix data types before any analysis.

In [ ]:
# --- Missing values ---
print("=== Missing values ===")
print("Transactions:\n", transactions.isnull().sum())
print("\nResponse:\n",   response.isnull().sum())

In [ ]:
transactions.dropna(inplace=True)
response.dropna(inplace=True)

In [ ]:
# --- Duplicates ---
print("Duplicate rows in transactions:", transactions.duplicated().sum())
print("Duplicate rows in response:   ", response.duplicated().sum())
transactions.drop_duplicates(inplace=True)
response.drop_duplicates(inplace=True)

In [ ]:
# --- Fix date format (DD-Mon-YY) ---
transactions["trans_date"] = pd.to_datetime(transactions["trans_date"], format="%d-%b-%y")
print(transactions.dtypes)
transactions.info()

### Outlier Detection — Z-Score on `tran_amount`

Values with |z| > 3 are flagged as statistical outliers.

In [ ]:
z_scores = np.abs(stats.zscore(transactions["tran_amount"]))
outlier_mask = z_scores > 3
print(f"Outliers detected: {outlier_mask.sum()}")
print("\nOutlier rows (if any):")
print(transactions[outlier_mask])

# Visualise distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(transactions["tran_amount"], bins=30, color="steelblue", edgecolor="white")
axes[0].set_title("Transaction Amount Distribution")
axes[0].set_xlabel("Amount ($)")
sns.boxplot(x=transactions["tran_amount"], ax=axes[1], color="steelblue")
axes[1].set_title("Boxplot — tran_amount")
plt.tight_layout()
plt.show()
print("\nNo outliers found; all values fall within 3 standard deviations of the mean.")

## 3. Data Preparation

Merge datasets and engineer time-based features needed for analysis.

In [ ]:
# Inner join keeps only customers present in both datasets
df = pd.merge(transactions, response, on="customer_id", how="inner")

# Time features
df["Year"]    = df["trans_date"].dt.year
df["Month"]   = df["trans_date"].dt.month
df["Quarter"] = df["trans_date"].dt.quarter

# Total spend per transaction (already at item level, but alias for clarity)
df["total_sales"] = df["tran_amount"]

print("Merged shape:", df.shape)
df.head()

## 4. Exploratory Data Analysis

### 4.1 Key Metrics

In [ ]:
total_rev = df["tran_amount"].sum()
avg_txn   = df["tran_amount"].mean()
total_cust = df["customer_id"].nunique()
resp_rate = response["response"].mean()

print(f"Total Revenue          : ${total_rev:,.0f}")
print(f"Average Transaction    : ${avg_txn:.2f}")
print(f"Unique Customers       : {total_cust:,}")
print(f"Campaign Response Rate : {resp_rate:.1%}  ({response['response'].sum()} of {len(response)})")

### 4.2 Revenue by Year

In [ ]:
yearly = df.groupby("Year")["tran_amount"].sum()
print(yearly.to_string())

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(yearly.index.astype(str), yearly.values, color="steelblue", edgecolor="white", width=0.6)
ax.bar_label(bars, fmt="$%,.0f", padding=4, fontsize=9)
ax.set_title("Revenue by Year")
ax.set_xlabel("Year")
ax.set_ylabel("Revenue ($)")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x/1e6:.1f}M"))
plt.tight_layout()
plt.show()

### 4.3 Monthly Revenue Trend (Time Series)

In [ ]:
df["YearMonth"] = df["trans_date"].dt.to_period("M")
monthly_ts = df.groupby("YearMonth")["tran_amount"].sum()
monthly_ts.index = monthly_ts.index.to_timestamp()

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(monthly_ts.index, monthly_ts.values, color="steelblue", linewidth=1.8)
ax.fill_between(monthly_ts.index, monthly_ts.values, alpha=0.15, color="steelblue")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.set_title("Monthly Revenue Trend (May 2011 – Mar 2015)")
ax.set_xlabel("Month")
ax.set_ylabel("Revenue ($)")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x:,.0f}"))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 4.4 Top Revenue Months (Seasonal Pattern)

In [ ]:
monthly_agg = df.groupby("Month")["tran_amount"].sum().sort_values(ascending=False)
month_names = {1:"Jan",2:"Feb",3:"Mar",4:"Apr",5:"May",6:"Jun",
               7:"Jul",8:"Aug",9:"Sep",10:"Oct",11:"Nov",12:"Dec"}
monthly_agg.index = monthly_agg.index.map(month_names)

fig, ax = plt.subplots(figsize=(10, 4))
colors = ["#e07b39" if i < 3 else "steelblue" for i in range(12)]
bars = ax.bar(monthly_agg.index, monthly_agg.values, color=colors, edgecolor="white")
ax.set_title("Revenue by Month (aggregated across all years)\nTop 3 highlighted: August, October, January")
ax.set_xlabel("Month")
ax.set_ylabel("Revenue ($)")
plt.tight_layout()
plt.show()

### 4.5 Top 10 Customers by Revenue

In [ ]:
top_customers = (
    df.groupby("customer_id")["tran_amount"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)
print(top_customers.to_string())

fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(top_customers.index[::-1], top_customers.values[::-1], color="steelblue", edgecolor="white")
ax.set_title("Top 10 Customers by Total Spend")
ax.set_xlabel("Total Spend ($)")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x:,.0f}"))
plt.tight_layout()
plt.show()

### 4.6 Campaign Response Distribution

In [ ]:
resp_counts = df["response"].value_counts().sort_index()
labels = ["No Response (0)", "Responded (1)"]
colors = ["#d9534f", "#5cb85c"]

fig, ax = plt.subplots(figsize=(6, 6))
wedges, texts, autotexts = ax.pie(
    resp_counts, labels=labels, autopct="%1.1f%%",
    colors=colors, startangle=140, explode=(0, 0.08)
)
for t in autotexts: t.set_fontsize(12)
ax.set_title("Customer Response Distribution")
plt.tight_layout()
plt.show()

## 5. Customer Segmentation — RFM Analysis

RFM (Recency, Frequency, Monetary) is a standard customer segmentation framework. Each customer receives a score from 1–4 on all three dimensions; the total score maps to a business segment.

In [ ]:
max_date = df["trans_date"].max()

rfm = df.groupby("customer_id").agg(
    recency  =("trans_date",  lambda x: (max_date - x.max()).days),
    frequency=("trans_date",  "count"),
    monetary =("tran_amount", "sum")
).reset_index()

# Score each dimension 1–4 via quartile ranking
rfm["R"] = pd.qcut(rfm["recency"],   4, labels=[4, 3, 2, 1]).astype(int)  # lower recency = better
rfm["F"] = pd.qcut(rfm["frequency"].rank(method="first"), 4, labels=[1, 2, 3, 4]).astype(int)
rfm["M"] = pd.qcut(rfm["monetary"],  4, labels=[1, 2, 3, 4]).astype(int)
rfm["RFM_Score"] = rfm["R"] + rfm["F"] + rfm["M"]

def assign_segment(score):
    if score >= 10: return "Champions"
    elif score >= 7: return "Loyal Customers"
    elif score >= 5: return "At Risk"
    else:            return "Lost"

rfm["Segment"] = rfm["RFM_Score"].apply(assign_segment)
rfm.head(10)

In [ ]:
seg_counts = rfm["Segment"].value_counts()
print(seg_counts.to_string())

palette = {"Champions":"#2ecc71","Loyal Customers":"#3498db","At Risk":"#e67e22","Lost":"#e74c3c"}
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(seg_counts.index, seg_counts.values,
              color=[palette[s] for s in seg_counts.index], edgecolor="white")
ax.bar_label(bars, padding=4, fontsize=10)
ax.set_title("RFM Customer Segmentation")
ax.set_xlabel("Segment")
ax.set_ylabel("Number of Customers")
plt.tight_layout()
plt.show()

### 5.1 Top Customer Behaviour Over Time

Tracking monthly spend of the top 5 customers reveals whether high-value customers are consistent or seasonal.

In [ ]:
top5_ids = rfm.nlargest(5, "monetary")["customer_id"]
top5_df  = df[df["customer_id"].isin(top5_ids)].copy()
top5_ts  = top5_df.groupby(["YearMonth", "customer_id"])["tran_amount"].sum().unstack()
top5_ts.index = top5_ts.index.to_timestamp()

fig, ax = plt.subplots(figsize=(13, 5))
top5_ts.plot(ax=ax, marker="o", markersize=4)
ax.set_title("Monthly Spend — Top 5 Customers")
ax.set_xlabel("Date")
ax.set_ylabel("Spend ($)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45)
plt.legend(title="Customer ID", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 6. Key Findings

- **Total revenue** generated across the dataset was **$8,122,062** from 124,963 transactions.
- **Peak months** by revenue: **August, October, and January** — suggesting back-to-school and holiday-adjacent purchasing peaks.
- **Revenue by year**: 2013 was the strongest year ($2.14M); 2015 data is partial (Jan–Mar only).
- **No statistical outliers** were found in transaction amounts (all values fall within 3 standard deviations).
- **Campaign response rate**: 9.4% of customers (647 of 6,884) responded to the marketing campaign.
- **RFM segmentation** revealed that **1,880 Champions** and **2,402 Loyal Customers** form the core revenue base — over 62% of all customers.

## 7. Recommendations

1. **Prioritise Champions and Loyal Customers** with exclusive offers and early access to new products.
2. **Re-engage At Risk customers** (1,518) with personalised win-back campaigns before they churn.
3. **Double down on August and October** promotions — these months consistently drive the highest revenue.
4. **Improve campaign targeting**: the 9.4% response rate indicates broad targeting; RFM segments should be used to personalise outreach.
5. **Investigate February dips**: revenue consistently drops in February across years — a targeted mid-winter promotion could counter this.

## 8. Conclusion

The analysis delivered actionable insights across revenue trends, seasonal patterns, and customer value distribution. The RFM model provides a repeatable framework for ongoing customer lifecycle management, enabling the retail chain to move from reactive reporting to proactive, segment-driven strategy.